# RAGAS Evaluation

This notebook evaluates the Agentic RAG pipeline using the [RAGAS framework](https://docs.ragas.io/).
Because we don't have a labeled ground-truth dataset, we will evaluate using:
- **Faithfulness**: Measures if the generated answer is grounded in the retrieved context (checking for hallucinations).
- **Answer Relevance**: Measures how directly the generated answer addresses the user's initial query.

In [1]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")


True

### 1. Generate Pipeline Outputs
We will run 3 diverse test queries representing our different pipeline branches (Direct LLM, Vectorstore, Web Search).

In [ ]:
from main import agentic_rag_app
from datasets import Dataset

def get_pipeline_data(query: str):
    initial_state = {
        "query": query,
        "original_query": query,
        "namespace": "default_namespace",
        "documents": [],
        "final_context_strips": [],
        "needs_web_search": False,
        "answer": "",
        "retries": 0,
        "route": ""
    }
    result = agentic_rag_app.invoke(initial_state)
    
    # lists for context strings
    context = result.get("final_context_strips", [])
    if not context:
        context = ["Default global knowledge context."]
        
    return result.get("answer", ""), context

queries = [
    "Tell me about the cross-modal architecture of RAG-Anything.",
    "Who won the latest Super Bowl?",
    "What is the specific dual-graph construction technique used? Explain step-by-step."
]

data = {"question": [], "answer": [], "contexts": []}

for q in queries:
    print(f"Evaluating query: {q}")
    ans, ctx = get_pipeline_data(q)
    data["question"].append(q)
    data["answer"].append(ans)
    data["contexts"].append(ctx)

# Create huggingface dataset formatting required by RAGAS
dataset = Dataset.from_dict(data)
dataset.to_pandas()

c:\Users\amanm\Desktop\learning\developer-chat-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index 'agenticrag' already exists


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8218.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating query: Tell me about the cross-modal architecture of RAG-Anything.
Evaluating query: Who won the latest Super Bowl?
Evaluating query: What is the specific dual-graph construction technique used? Explain step-by-step.


,question,answer,contexts
0,Tell me about the cross-modal architecture of ...,**RAG‑Anything’s cross‑modal architecture** is...,[specialized layout-aware parsing modules that...
1,Who won the latest Super Bowl?,"The Kansas City Chiefs won Super Bowl LVIII, d...",[[Web Search Results]: Discover the NFLplayert...
2,What is the specific dual-graph construction t...,**Constructing the Dual of a Planar (or More G...,"[[Web Search Results]: Ingraphtheory,aplanargr..."


### 2. Configure RAGAS Custom Models
To avoid OpenAI API costs by default, we configure RAGAS to use our Groq LLM and HuggingFace Local Embeddings.

In [3]:
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings
from src.generation.generator import llm

# Wrap our models for RAGAS compatibility
ragas_llm = LangchainLLMWrapper(llm)
ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))

# Explicitly assign models to metrics (RAGAS 0.1.X pattern)
faithfulness.llm = ragas_llm
answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = ragas_emb

C:\Users\amanm\AppData\Local\Temp\ipykernel_12492\1970055348.py:1: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\amanm\AppData\Local\Temp\ipykernel_12492\1970055348.py:1: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\amanm\AppData\Local\Temp\ipykernel_12492\1970055348.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'

### 3. Run RAGAS Evaluation

In [4]:
from ragas import evaluate

print("Running metrics...")
results = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=ragas_emb,
)

df_results = results.to_pandas()
print("Evaluation complete!")
df_results[["question", "faithfulness", "answer_relevancy"]]

Running metrics...


Evaluating: 100%|██████████| 6/6 [02:35<00:00, 25.96s/it]


Evaluation complete!


KeyError: "['question'] not in index"